============================================================
Kaggle Notebook: Topology Ablation Re-Run at max_seq_len=10,000
============================================================
Run ONE notebook per variant (no_graph, undirected, dense) --
duplicate this notebook 3x and change MODE at the top of each,
or run sequentially in one notebook across 3 separate commits.

"directed" does NOT need to be re-run here: your existing
graph_dkt_10k_fold{0-4}.pt weights already ARE the directed
variant (adjacency-mode directed is the default/baseline path
in train_dkt.py -- confirmed directly against source).

DKT does NOT need to be re-run either: DKT has no adjacency
input, so dkt_10k_fold{0-4}.pt is identical across all variants.

Expected runtime per variant: ~9.3h (5 folds x 50 epochs, based
on your actual directed-variant retrain timing). Fits in one
12h Kaggle session with margin -- but if it runs long, use
--start-fold/--end-fold to resume across sessions exactly like
the primary retrain.

Before running: upload two datasets and attach both via
"+ Add Data":
  1. 'dkt-10k-weights-5fold'      containing dkt_10k_fold{0-4}.pt
  2. 'directed-10k-weights-5fold' containing graph_dkt_10k_fold{0-4}.pt
============================================================


In [ ]:
MODE = "no_graph"  # change to "undirected" or "dense" for the other two notebooks


### CELL 0 — Install dependencies (same fix as before: --no-deps to protect CUDA torch)


In [ ]:
import torch

print(f"CUDA available BEFORE any installs: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "Check accelerator setting -- should be GPU T4 x2 or x1."
print(f"Device: {torch.cuda.get_device_name(0)}")

try:
    import torch_geometric
    print("torch_geometric already installed!")
except ImportError:
    !pip install torch-geometric --no-deps -q

!pip install scikit-learn pandas tqdm -q

print(f"\nCUDA available AFTER installs: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA broke during pip install despite --no-deps! Restart the kernel."


### CELL 1 — Copy training.csv from Kaggle input


In [ ]:
import os, shutil

input_dir = "/kaggle/input"
training_csv = None
for root, dirs, files in os.walk(input_dir):
    if "training.csv" in files:
        training_csv = os.path.join(root, "training.csv")
        break

if training_csv is None:
    print("ERROR: training.csv not found in Kaggle input! Add it via Add Data -> Upload.")
    import sys; sys.exit(1)

print(f"Found: {training_csv} ({os.path.getsize(training_csv) / 1e6:.1f} MB)")


### CELL 2 — Clone repo and copy training data


In [ ]:
!git clone https://github.com/MannPatel1236/CP-Coach.git /kaggle/working/CP-Coach

data_dir = "/kaggle/working/CP-Coach/backend/data"
os.makedirs(data_dir, exist_ok=True)
shutil.copy(training_csv, os.path.join(data_dir, "training.csv"))
print(f"Copied training.csv to {data_dir}")


### CELL 2b — Restore DKT weights (identical across all variants, never retrained here)


In [ ]:
%cd /kaggle/working/CP-Coach/backend
weights_dir = "/kaggle/working/CP-Coach/backend/weights"
os.makedirs(weights_dir, exist_ok=True)

import glob
found_dkt = False
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.startswith('dkt_10k_fold') and f.endswith('.pt'):
            shutil.copy(os.path.join(root, f), weights_dir)
            print(f"Restored: {f} from {root}")
            found_dkt = True

if not found_dkt:
    raise FileNotFoundError(
        "DKT weights dataset not found. Upload dkt_10k_fold{0-4}.pt as a private "
        "Kaggle dataset named 'dkt-10k-weights-5fold', then Add Data -> Your Datasets."
    )


### CELL 3 — Smoke test (3 epochs, 1 fold) before committing to the full run


In [ ]:
!python3 -m training.train_dkt \
  --data data/training.csv --model graph_dkt \
  --adjacency-mode {MODE} \
  --epochs 3 --folds 1 --out weights/ablation_{MODE}_10k.pt \
  --batch 16 --device cuda --max-seq-len 10000
# Watch for: no OOM, no traceback, and a printed AUC around 0.85-0.96 after 3 epochs
# (won't be converged yet, just confirms the pipeline runs end-to-end at seq_len=10000).


### CELL 4 — Full retrain: this variant, 5 folds x 50 epochs, max_seq_len=10000


In [ ]:
# Expected ~9.3h. If it risks exceeding a 12h session, add
# --start-fold N --end-fold N to split across multiple commits,
# exactly like the primary DKT/Graph-DKT retrain.
!python3 -m training.train_dkt \
  --data data/training.csv --model graph_dkt \
  --adjacency-mode {MODE} \
  --epochs 50 --folds 5 --out weights/ablation_{MODE}_10k.pt \
  --batch 16 --device cuda --max-seq-len 10000


### CELL 5 — Restore the existing "directed" variant (already trained -- your graph_dkt_10k_fold*.pt)


In [ ]:
found_directed = False
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.startswith('graph_dkt_10k_fold') and f.endswith('.pt'):
            shutil.copy(os.path.join(root, f), weights_dir)
            print(f"Restored (directed baseline): {f} from {root}")
            found_directed = True

if not found_directed:
    raise FileNotFoundError(
        "Directed-variant weights not found. Upload graph_dkt_10k_fold{0-4}.pt as a "
        "private Kaggle dataset named 'directed-10k-weights-5fold'."
    )


### CELL 6 — Rename this variant's weights so eval_folds.py picks them up as "graph_dkt"


In [ ]:
import shutil

for i in range(5):
    shutil.copy(f"weights/ablation_{MODE}_10k_fold{i}.pt", f"weights/graph_dkt_fold{i}.pt")
    shutil.copy(f"weights/dkt_10k_fold{i}.pt", f"weights/dkt_fold{i}.pt")
    print(f"Prepared weights for fold {i}")

!python3 -m training.eval_folds --segment-by-length --device cpu --max-seq-len 10000 --folds 5
# SAVE THIS OUTPUT -- it's the AUC/P@5 row for this variant at max_seq_len=10000.


### CELL 7 — Package results for download


In [ ]:
import zipfile

with zipfile.ZipFile("/kaggle/working/results.zip", "w") as z:
    for f in glob.glob(f"weights/ablation_{MODE}_10k_fold*.pt"):
        z.write(f)
        print(f"Packed: {f}")

print(f"\nDone! Download /kaggle/working/results.zip -- this is the '{MODE}' variant at max_seq_len=10000.")
print("Repeat this whole notebook (change MODE at the top) for the other two variants.")
